# Enhanced Machine Learning Training Workflow

This notebook provides a complete workflow for training machine learning models using our advanced training pipeline. It includes:

1. Modern Architecture Features:
   - ALiBi, RoPE positional embeddings
   - MQA/GQA attention mechanisms
   - SwiGLU and RMSNorm
   - Flash Attention

2. Parameter-Efficient Fine-Tuning (PEFT):
   - LoRA/QLoRA
   - Adapters
   - Prefix Tuning

3. Advanced Distributed Training:
   - FSDP (Fully Sharded Data Parallel)
   - DeepSpeed ZeRO optimizations
   - PyTorch 2.0 compilation

4. Inference Optimization:
   - Continuous batching
   - Speculative decoding
   - KV cache management

## Setup Environment

First, we'll clone the repository and install the required dependencies.

In [ ]:
# Clone the repository
!git clone https://github.com/backup-bdg-6/datasets.git ml-training-pipeline
%cd ml-training-pipeline

In [ ]:
# Install required dependencies
!pip install -r requirements.txt huggingface_hub

# Install additional dependencies for advanced features
!pip install "ray[tune]" deepspeed bitsandbytes flash-attn
!pip install "transformers>=4.30.0" "datasets>=2.10.0" "accelerate>=0.20.0" "torch>=2.0.0"

## Set Hugging Face API Token

Some datasets require authentication. Let's set up the Hugging Face API token.

In [ ]:
import os
from huggingface_hub import login, HfApi

# Set your Hugging Face API token
# For Kaggle, you can use the Kaggle secrets
# For Google Colab, you can use the following code to input your token securely
from getpass import getpass

# Input your token securely
HUGGINGFACE_TOKEN = getpass("Enter your Hugging Face token: ")

# Login to HuggingFace Hub
if HUGGINGFACE_TOKEN:
    login(token=HUGGINGFACE_TOKEN)
    print("Successfully logged in to Hugging Face Hub")
else:
    print("No Hugging Face token provided. Some datasets may not be accessible.")

## Import Required Modules

Now, let's import all the necessary modules for our enhanced workflow.

In [ ]:
import os
import sys
import logging
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
import time
from typing import Dict, List, Optional, Union, Any, Tuple, Callable

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# Import our modules
from src.data.loaders import DatasetLoader
from src.data.preprocessors import DataPreprocessor
from src.data.augmentation import TextAugmenter, augment_dataset
from src.model.architecture import create_model_from_config
from src.model.training import Trainer, TrainingArguments
from src.model.distributed_training import train_distributed, DeepSpeedConfig, DistributedTrainer
from src.model.advanced_distributed import train_with_fsdp, FSDPConfig, setup_distributed
from src.model.peft import apply_peft_to_model, prepare_for_qlora, LoRAConfig, AdapterConfig, PrefixTuningConfig
from src.model.inference_optimization import ContinuousBatchingServer, SpeculativeDecoder, KVCacheManager
from src.utils.tokenization import get_tokenizer
from src.utils.hyperparameter_tuning import optimize_hyperparameters
from src.utils.model_evaluation import evaluate_model
from src.model.optimization import optimize_model

# Check for optional dependencies
try:
    import torch._dynamo
    TORCH_COMPILE_AVAILABLE = True
    print("PyTorch compilation (torch.compile) is available.")
except ImportError:
    TORCH_COMPILE_AVAILABLE = False
    print("PyTorch compilation (torch.compile) is not available. Please upgrade to PyTorch 2.0+")

try:
    import deepspeed
    DEEPSPEED_AVAILABLE = True
    print("DeepSpeed is available.")
except ImportError:
    DEEPSPEED_AVAILABLE = False
    print("DeepSpeed is not available. Advanced distributed training features will be limited.")

try:
    import bitsandbytes as bnb
    BITSANDBYTES_AVAILABLE = True
    print("Bitsandbytes is available. QLoRA and 4/8-bit quantization is supported.")
except ImportError:
    BITSANDBYTES_AVAILABLE = False
    print("Bitsandbytes is not available. QLoRA and 4/8-bit quantization will not be available.")

try:
    import flash_attn
    FLASH_ATTENTION_AVAILABLE = True
    print("Flash Attention is available.")
except ImportError:
    FLASH_ATTENTION_AVAILABLE = False
    print("Flash Attention is not available. Using standard attention implementation.")

## Configuration

Let's create an enhanced configuration for our training workflow with all advanced features.

In [ ]:
# Create a configuration directory if it doesn't exist
!mkdir -p config

In [ ]:
%%writefile config/enhanced_training_config.yaml
# Enhanced configuration with all advanced features

# Project information
project_name: "enhanced_transformer_training"
output_dir: "./output_enhanced"
seed: 42

# Model configuration with advanced architecture settings
model:
  type: "decoder_only_transformer"
  size: "small"  # Using small model for notebooks
  sizes:
    small:
      n_layers: 6
      n_heads: 8
      d_model: 512
      d_ff: 2048
      max_seq_length: 1024
  dropout: 0.1
  attention:
    causal: true
    rotary_embedding: true
  
  # Advanced architecture settings
  architecture:
    # Use ALiBi instead of RoPE for better length extrapolation
    position_embeddings: "alibi"  # Options: "rotary", "alibi", "learned", "none"
    
    # Use Grouped-Query Attention for efficiency
    attention_type: "gqa"  # Options: "mha", "mqa", "gqa"
    kv_heads: 2  # Number of KV heads for GQA (n_heads / kv_heads queries per KV head)
    
    # Use RMSNorm instead of LayerNorm
    norm_type: "rms_norm"  # Options: "layer_norm", "rms_norm"
    normalization_strategy: "pre_norm"
    
    # Use SwiGLU for better performance
    ffn_type: "swiglu"  # Options: "mlp", "swiglu", "geglu"
    
    # Turn on flash attention if available
    use_flash_attention: true
    
    # Use different initialization
    initialization:
      method: "kaiming_normal"
  
  # PyTorch 2.0+ compilation (significantly speeds up training and inference)
  compile:
    enabled: true
    mode: "reduce-overhead"
  
  # Use gradient checkpointing to save memory
  gradient_checkpointing: true

# Tokenizer configuration
tokenizer:
  type: "huggingface"
  name: "gpt2"
  vocab_size: 50257
  max_length: 1024
  padding_side: "right"
  truncation_side: "right"
  add_bos_token: true
  add_eos_token: true

# Parameter-Efficient Fine-Tuning (PEFT) configuration
peft:
  enabled: true
  method: "lora"  # Options: lora, qlora, adapter, prefix_tuning
  
  # LoRA configuration
  lora:
    r: 16
    alpha: 32
    dropout: 0.05
    target_modules:
      - "q_proj"
      - "k_proj"
      - "v_proj"
      - "o_proj"
      - "gate_proj"
      - "up_proj"
      - "down_proj"
    bias: "none"
    use_rslora: false
  
  # QLoRA configuration
  qlora:
    r: 16
    alpha: 32
    dropout: 0.05
    target_modules:
      - "q_proj"
      - "k_proj"
      - "v_proj"
      - "o_proj"
      - "gate_proj"
      - "up_proj"
      - "down_proj"
    bias: "none"
    bits: 4
    double_quant: true
    bnb_blocksize: 64
    bnb_compute_dtype: "float16"

# Training configuration
training:
  active_stage: "lora_finetune"
  stages:
    - name: "lora_finetune"
      epochs: 3
      datasets:
        - name: "imdb"
          split: "train"
          streaming: false
          max_samples: 5000
        - name: "imdb"
          split: "test"
          streaming: false
          max_samples: 1000
      learning_rate:
        initial: 2.0e-4
        min: 1.0e-6
        schedule: "cosine"
        warmup_steps: 200
      # Use LoRA for efficient fine-tuning
      peft_method: "lora"
      peft_config:
        r: 16
        alpha: 32
        dropout: 0.05
  
  # Optimizer settings
  optimizer:
    name: "adamw"
    weight_decay: 0.01
    beta1: 0.9
    beta2: 0.999
    eps: 1.0e-8
    use_8bit: true  # Use 8-bit optimizer for memory efficiency
  
  # Mixed precision settings
  mixed_precision: "bf16"  # Use BF16 for better numerical stability
  gradient_clipping: 1.0
  
  # Checkpointing settings
  checkpointing:
    save_steps: 100
    keep_last_n: 2
    save_optimizer_state: true
    merge_lora_weights: true  # Merge LoRA weights at the end
    export_formats:
      - "safetensors"
  
  # Evaluation settings
  evaluation:
    eval_steps: 100
    early_stopping:
      enabled: true
      patience: 3
      metric: "eval_loss"
      mode: "min"

# Distributed training configuration
distributed_training:
  strategy: "deepspeed"
  
  # DeepSpeed configuration
  deepspeed:
    enabled: true
    zero_stage: 2
    offload_optimizer: false
    offload_param: false
    pin_memory: true
    contiguous_gradients: true
  
  # FSDP configuration (for single node multi-GPU)
  fsdp:
    enabled: false
    sharding_strategy: "full"
    mixed_precision: "bf16"
    cpu_offload: false
    auto_wrap_policy: "transformer"

# Data processing configuration
data_processing:
  preprocessing:
    normalize_unicode: true
    normalize_whitespace: true
    remove_html: true
    min_length: 10
    max_length: 1024
  
  augmentation:
    enabled: true
    techniques:
      - name: "synonym_replacement"
        probability: 0.1
      - name: "random_deletion"
        probability: 0.05
  
  batching:
    train_batch_size: 8
    eval_batch_size: 8
    gradient_accumulation_steps: 8  # Effectively 64 batch size
    dynamic_batching: true  # Dynamic batching based on sequence lengths

# Inference optimization configuration
inference_optimization:
  # General inference settings
  dtype: "bfloat16"
  device: "cuda"
  compile: true
  
  # KV cache management
  kv_cache:
    enabled: true
    pruning_strategy: "sliding_window"
    window_size: 2048
  
  # Continuous batching for efficient token generation
  continuous_batching:
    enabled: true
    max_batch_size: 32
    max_waiting_tokens: 8192
    streaming: true
  
  # Speculative decoding with a smaller model
  speculative_decoding:
    enabled: false  # Requires a draft model
    num_speculative_tokens: 4

# Evaluation configuration
evaluation:
  metrics:
    - "loss"
    - "perplexity"
    - "accuracy"
    - "bleu"
    - "rouge"
  
  generation:
    max_length: 128
    num_beams: 4
    temperature: 1.0
    top_p: 0.9
    top_k: 50
    do_sample: true
    repetition_penalty: 1.1

# HuggingFace Hub integration
huggingface_hub:
  push_to_hub: false
  private: false

## Load Configuration

Now, let's load the enhanced configuration from the YAML file.

In [ ]:
def load_config(config_path: str) -> Dict:
    """Load configuration from YAML file."""
    try:
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
        return config
    except Exception as e:
        logger.error(f"Error loading configuration: {e}")
        raise

# Load configuration
config_path = "config/enhanced_training_config.yaml"
config = load_config(config_path)

# Set random seed
torch.manual_seed(config['seed'])
np.random.seed(config['seed'])

# Create output directory
os.makedirs(config['output_dir'], exist_ok=True)

## Modern Architecture Features

Let's explore the modern architecture features we've implemented:

In [ ]:
# Explore the architecture configuration
print("\n===== Modern Architecture Features =====\n")
print(f"Position Embeddings: {config['model']['architecture']['position_embeddings']}")
print(f"Attention Type: {config['model']['architecture']['attention_type']}")
print(f"KV Heads (for GQA): {config['model']['architecture']['kv_heads']}")
print(f"Normalization: {config['model']['architecture']['norm_type']}")
print(f"Feed-Forward Network: {config['model']['architecture']['ffn_type']}")
print(f"Flash Attention: {config['model']['architecture']['use_flash_attention']}")
print(f"Gradient Checkpointing: {config['model']['gradient_checkpointing']}")
print(f"PyTorch Compilation: {config['model']['compile']['enabled']}")

# Create a sample model to test the architecture
print("\nCreating sample model with modern architecture features...")
model = create_model_from_config(config)
print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")

## Parameter-Efficient Fine-Tuning (PEFT)

Let's explore the PEFT methods we've implemented for memory-efficient fine-tuning:

In [ ]:
# Explore PEFT configuration
print("\n===== Parameter-Efficient Fine-Tuning =====\n")
print(f"PEFT Enabled: {config['peft']['enabled']}")
print(f"PEFT Method: {config['peft']['method']}")

# Get LoRA configuration
lora_config = config['peft']['lora']
print(f"\nLoRA Configuration:")
print(f"  Rank (r): {lora_config['r']}")
print(f"  Alpha: {lora_config['alpha']}")
print(f"  Target Modules: {', '.join(lora_config['target_modules'])}")

# Apply LoRA to the model
if BITSANDBYTES_AVAILABLE and config['peft']['enabled']:
    print("\nApplying LoRA to the model...")
    trainable_params_before = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # Apply PEFT
    peft_config = config['peft'][config['peft']['method']] 
    model = apply_peft_to_model(
        model=model,
        peft_config=peft_config,
        peft_type=config['peft']['method'],
        adapter_name="default"
    )
    
    trainable_params_after = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters before LoRA: {trainable_params_before:,} ({trainable_params_before/total_params:.2%})")
    print(f"Trainable parameters after LoRA: {trainable_params_after:,} ({trainable_params_after/total_params:.2%})")
    print(f"Parameter reduction: {trainable_params_before/trainable_params_after:.1f}x")
    
    # Show the model's trainable parameters
    print("\nTrainable parameters:")
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(f"  {name}: {param.shape}")
else:
    print("\nSkipping LoRA application: bitsandbytes not available or PEFT not enabled in config")

## Data Loading and Preprocessing

Now let's load and preprocess datasets for training using our pipeline:

In [ ]:
# Get active stage
active_stage = config['training']['active_stage']
logger.info(f"Active training stage: {active_stage}")

# Get stage configuration
stage_config = None
for stage in config['training']['stages']:
    if stage['name'] == active_stage:
        stage_config = stage
        break

if stage_config is None:
    raise ValueError(f"Training stage {active_stage} not found in configuration")

# Initialize dataset loader with HuggingFace token if available
dataset_loader = DatasetLoader(config_path, huggingface_token=HUGGINGFACE_TOKEN)

# Load datasets
train_datasets = []
for dataset_config in stage_config['datasets']:
    if dataset_config.get('split') == 'train':
        dataset = dataset_loader.load_dataset(
            dataset_config['name'],
            subset=dataset_config.get('subset'),
            split='train',
            streaming=dataset_config.get('streaming', False),
            max_samples=dataset_config.get('max_samples')
        )
        print(f"Loaded training dataset '{dataset_config['name']}' with {len(dataset)} examples")
        train_datasets.append(dataset)

eval_datasets = []
for dataset_config in stage_config['datasets']:
    if dataset_config.get('split') == 'test':
        dataset = dataset_loader.load_dataset(
            dataset_config['name'],
            subset=dataset_config.get('subset'),
            split='test',
            streaming=dataset_config.get('streaming', False),
            max_samples=dataset_config.get('max_samples')
        )
        print(f"Loaded test dataset '{dataset_config['name']}' with {len(dataset)} examples")
        eval_datasets.append(dataset)

# Initialize data preprocessor
preprocessor = DataPreprocessor(config)

# Preprocess datasets
train_datasets = [preprocessor.process_dataset(ds) for ds in train_datasets]
eval_datasets = [preprocessor.process_dataset(ds) for ds in eval_datasets]

In [ ]:
# Get tokenizer
tokenizer = get_tokenizer(config['tokenizer'])

# Print tokenizer details
print(f"Vocabulary size: {len(tokenizer)}")
print(f"Max length: {tokenizer.model_max_length}")
print(f"BOS token: {tokenizer.bos_token} ({tokenizer.bos_token_id})")
print(f"EOS token: {tokenizer.eos_token} ({tokenizer.eos_token_id})")
print(f"PAD token: {tokenizer.pad_token} ({tokenizer.pad_token_id})")

# Look at a sample from the dataset
if train_datasets:
    sample = train_datasets[0][0]
    print("\nSample from dataset:")
    for key, value in sample.items():
        if isinstance(value, str) and len(value) > 100:
            print(f"{key}: {value[:100]}...")
        else:
            print(f"{key}: {value}")

## Train a Model with Modern Features

Let's train a model using our enhanced pipeline with modern features:

In [ ]:
# Create training arguments
training_args = TrainingArguments(config, active_stage)

# Create trainer with the model we set up earlier
trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_datasets[0] if train_datasets else None,
    eval_dataset=eval_datasets[0] if eval_datasets else None,
    args=training_args
)

# Train for a small number of steps (for demonstration)
print("Training model for a small number of steps...")
training_args.epochs = 1  # Override for notebook demonstration
trainer.train()

## Merge LoRA Weights

After training with LoRA, we can merge the weights back into the base model for deployment:

In [ ]:
if config['peft']['enabled'] and hasattr(model, 'merge_peft_weights'):
    print("Merging LoRA weights into base model...")
    model.merge_peft_weights()
    print("Weights merged successfully")
    
    # Show that all parameters are now non-trainable
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters after merging: {trainable_params}")

## Optimize Model for Inference

Now, let's optimize the model for inference using our advanced techniques:

In [ ]:
# Optimize model for inference
print("\n===== Inference Optimization =====\n")

# First, export the model for inference
inference_dir = os.path.join(config['output_dir'], 'inference_ready')
os.makedirs(inference_dir, exist_ok=True)

# Save the model and tokenizer
model.save_pretrained(inference_dir)
tokenizer.save_pretrained(inference_dir)
print(f"Saved model and tokenizer to {inference_dir}")

# Get inference configuration
inference_config = config['inference_optimization']

# Apply optional optimizations based on the configuration
print("\nApplying inference optimizations:")
print(f"  Data type: {inference_config['dtype']}")
print(f"  Continuous batching: {inference_config['continuous_batching']['enabled']}")
print(f"  KV cache optimization: {inference_config['kv_cache']['enabled']}")
print(f"  Speculative decoding: {inference_config['speculative_decoding']['enabled']}")

# Demonstrate continuous batching
if inference_config['continuous_batching']['enabled']:
    print("\nSetting up continuous batching server for efficient inference...")
    
    # Create the server but don't actually start it (just show the setup)
    server_config = {
        "max_batch_size": inference_config['continuous_batching']['max_batch_size'],
        "max_waiting_tokens": inference_config['continuous_batching']['max_waiting_tokens'],
        "streaming": inference_config['continuous_batching']['streaming'],
        "device": inference_config['device'],
        "precision": inference_config['dtype']
    }
    print(f"Server configured with: {server_config}")
    
    # Show the API for adding requests and getting results
    print("\nAPI for using the continuous batching server:")
    print("""
    # Start the server
    server = ContinuousBatchingServer(model=model, tokenizer=tokenizer, **server_config)
    server.start()
    
    # Add requests
    request_id = server.add_request(GenerationRequest(
        request_id="req1",
        prompt="Explain the concept of machine learning",
        max_new_tokens=512
    ))
    
    # Get results (blocking until complete)
    results = server.get_results(request_id)
    
    # Stream results in real-time
    for chunk in server.stream_results(request_id):
        print(chunk["new_text"], end="", flush=True)
    """)

## Test Generation with the Model

Let's test text generation with our trained and optimized model:

In [ ]:
# Function to generate text
def generate_text(prompt, max_length=100, temperature=0.7, top_p=0.9, top_k=50):
    """Generate text from a prompt."""
    # Set model to evaluation mode
    model.eval()
    
    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate text
    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            max_length=max_length,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    # Decode the generated text
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return generated_text

# Test generation with the model
prompt = "This movie was really"
print(f"Prompt: {prompt}")
generated_text = generate_text(prompt)
print(f"Generated text: {generated_text}")

## Export and Share Model

Finally, let's export the model for sharing and deployment:

In [ ]:
# Save model and tokenizer
final_output_dir = os.path.join(config['output_dir'], 'final_model')
os.makedirs(final_output_dir, exist_ok=True)

print(f"Saving final model to {final_output_dir}")
model.save_pretrained(final_output_dir)
tokenizer.save_pretrained(final_output_dir)

# Save training config
with open(os.path.join(final_output_dir, 'config.yaml'), 'w') as f:
    yaml.dump(config, f)

print("Model, tokenizer, and config saved successfully")

# Instructions for using the model from the saved directory
print("\nTo load and use the saved model:")
print("""
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load the model and tokenizer
model = AutoModelForCausalLM.from_pretrained("path/to/final_model")
tokenizer = AutoTokenizer.from_pretrained("path/to/final_model")

# Use the model for inference
inputs = tokenizer("Your prompt here", return_tensors="pt")
outputs = model.generate(**inputs, max_length=100)
text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(text)
""")

## Optional: Push to HuggingFace Hub

You can push your model to HuggingFace Hub for sharing and deployment:

In [ ]:
if HUGGINGFACE_HUB_AVAILABLE and HUGGINGFACE_TOKEN:
    # Set model ID - change this to your username/model-name
    model_id = "your-username/enhanced-transformer-model"
    
    print(f"Pushing model to HuggingFace Hub with ID: {model_id}")
    
    # Create or get repo
    api = HfApi()
    repo_url = api.create_repo(repo_id=model_id, exist_ok=True, token=HUGGINGFACE_TOKEN)
    print(f"Repository URL: {repo_url}")
    
    # Push model and tokenizer
    model.push_to_hub(model_id, use_auth_token=HUGGINGFACE_TOKEN)
    tokenizer.push_to_hub(model_id, use_auth_token=HUGGINGFACE_TOKEN)
    
    print(f"Model and tokenizer pushed successfully to {model_id}")
    print(f"Access your model at: https://huggingface.co/{model_id}")
else:
    print("Skipping HuggingFace Hub upload: Token not provided or hub library not available")

## Conclusion

In this notebook, we've demonstrated a complete ML training workflow with our enhanced features:

1. Modern architecture capabilities (ALiBi, GQA, RMSNorm, SwiGLU, Flash Attention)
2. Parameter-efficient fine-tuning with LoRA
3. Advanced training optimizations including mixed precision and gradient checkpointing
4. Inference optimizations with continuous batching
5. Model export and sharing capabilities

This workflow can be applied to larger models and datasets on more powerful hardware, and scales efficiently with distributed training.